In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import time

In [79]:
# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("llm-jp/llm-jp-1.3b-v1.0")
model = AutoModelForCausalLM.from_pretrained("llm-jp/llm-jp-1.3b-v1.0", dtype=torch.float16)

# to GPU
if torch.cuda.is_available():
    model = model.to("cuda")
print("Model loaded to device:", model.device)

Loading weights: 100%|██████████| 293/293 [00:00<00:00, 18813.72it/s]


Model loaded to device: cpu


/home/shinnosuke/miniconda3/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## greedy search

In [ ]:
text = "慶應義塾大学"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    output = model.generate(
        tokenized_input,
        max_new_tokens=5, # 生成するトークン数
        num_beams=1, # beam searchのビーム幅
        do_sample=False,
        return_dict_in_generate=True, 
        output_scores=True,
    )

end_time = time.perf_counter()

print(tokenizer.decode(output.sequences[0])) # ID を文字列に変換して出力
print(f"Time taken: {end_time - start_time:.2f} seconds")

In [ ]:
# 各トークンの遷移確率を計算
transition_scores = model.compute_transition_scores(
    output.sequences, output.scores, normalize_logits=True
)

# 入力部分の長さを取得（生成されたトークンだけをループするため）
input_length = tokenized_input.shape[1]
generated_tokens = output.sequences[0][input_length:]

# 各トークンとその確率を表示
total_log_prob = 0.0
for tok, score in zip(generated_tokens, transition_scores[0]):
    token_str = tokenizer.decode(tok)
    # scoreは対数確率（log probability）なので、expをかけて通常の確率に戻す
    prob = torch.exp(score).item() * 100
    print(f"Token: {token_str:<10} | Probability: {prob:.2f}%")
    total_log_prob += score.item()  # 対数確率を足し合わせる

print(f"Total log probability of generated sequence: {total_log_prob:.4f}")

## Beam search

In [ ]:
text = "慶應義塾大学"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    output = model.generate(
        tokenized_input,
        max_new_tokens=5, # 生成するトークン数
        num_beams=5, # beam searchのビーム幅
        do_sample=False,
        return_dict_in_generate=True, 
        output_scores=True,
    )

end_time = time.perf_counter()

print(tokenizer.decode(output.sequences[0])) # ID を文字列に変換して出力
print(f"Time taken: {end_time - start_time:.2f} seconds")

In [ ]:
# 各トークンの遷移確率を計算
transition_scores = model.compute_transition_scores(
    output.sequences, output.scores, normalize_logits=True
)

# 入力部分の長さを取得（生成されたトークンだけをループするため）
input_length = tokenized_input.shape[1]
generated_tokens = output.sequences[0][input_length:]

# 各トークンとその確率を表示
total_log_prob = 0.0
for tok, score in zip(generated_tokens, transition_scores[0]):
    token_str = tokenizer.decode(tok)
    # scoreは対数確率（log probability）なので、expをかけて通常の確率に戻す
    prob = torch.exp(score).item() * 100
    print(f"Token: {token_str:<10} | Probability: {prob:.2f}%")
    total_log_prob += score.item()  # 対数確率を足し合わせる

print(f"Total log probability of generated sequence: {total_log_prob:.4f}")

## 最尤推定は反復を生じさせやすい

In [ ]:
text = "慶應義塾大学"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        tokenized_input,
        max_new_tokens=300, # 生成するトークン数
        num_beams=1, # beam searchのビーム幅
        do_sample=False,
    )

print(tokenizer.decode(output[0])) # ID を文字列に変換して出力

## Ancestral sampling

In [45]:
text = "明日"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

# ancestral sampling で 5 回サンプリング
num_samples = 5
samples = []

with torch.no_grad():
    for _ in range(num_samples):
        output = model.generate(
            tokenized_input,
            max_new_tokens=5, # 生成するトークン数
            do_sample=True,
        )
        samples.append(output)

for i, sample in enumerate(samples):
    print(f"Sample {i+1}: {tokenizer.decode(sample[0])}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to 

Sample 1: 明日、5月7日
Sample 2: 明日も素敵に☆2
Sample 3: 明日に向かいて |
Sample 4: 明日は、朝9時
Sample 5: 明日はいよいよ2日目


## Top-k sampling, Top-p sampling

In [73]:
# text に続く単語の確率分布を計算．確率の高い順に表示
text = "明日は"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        tokenized_input,
        max_new_tokens=1, # 生成するトークン数
        do_sample=True,
        return_dict_in_generate=True, 
        output_scores=True,
    )
# 表示
next_token_logits = output.scores[0]
next_token_probs = torch.softmax(next_token_logits, dim=-1)
top_k = 10
top_k_probs, top_k_indices = torch.topk(next_token_probs, top_k)
top_k_probs, top_k_indices  = top_k_probs[0], top_k_indices[0]

cumulative_prob = torch.cumsum(top_k_probs, dim=0)
# 確率の高い順に上位 10 個のトークンを表示（確率と累積確率も表示）
for i in range(top_k):
   print(f"Token: {tokenizer.decode(top_k_indices[i]):4s}, Prob: {top_k_probs[i].item():.4f}, Cumulative Prob: {cumulative_prob[i].item():.4f}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.


Token: 、   , Prob: 0.2234, Cumulative Prob: 0.2234
Token: いよいよ, Prob: 0.0714, Cumulative Prob: 0.2947
Token: 何   , Prob: 0.0438, Cumulative Prob: 0.3385
Token:     , Prob: 0.0349, Cumulative Prob: 0.3735
Token: 「   , Prob: 0.0341, Cumulative Prob: 0.4076
Token: お   , Prob: 0.0313, Cumulative Prob: 0.4389
Token: 雨   , Prob: 0.0288, Cumulative Prob: 0.4677
Token: 明日  , Prob: 0.0276, Cumulative Prob: 0.4954
Token: 朝   , Prob: 0.0248, Cumulative Prob: 0.5201
Token: また  , Prob: 0.0215, Cumulative Prob: 0.5417


In [74]:
text = "明日"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

# ancestral sampling で 5 回サンプリング
num_samples = 5
samples = []

with torch.no_grad():
    for _ in range(num_samples):
        output = model.generate(
            tokenized_input,
            max_new_tokens=5, # 生成するトークン数
            top_k=10, # 上位 k 個のトークンからサンプリング
            #top_p=0.9, # 上位 p% のトークンからサンプリング
            do_sample=True,
        )
        samples.append(output)

for i, sample in enumerate(samples):
    print(f"Sample {i+1}: {tokenizer.decode(sample[0])}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to 

Sample 1: 明日への架け橋」
Sample 2: 明日のお天気は？
Sample 3: 明日は、お友達と
Sample 4: 明日はお仕事でした
Sample 5: 明日のために |


## Temperature sampling

In [77]:
text = "明日"
tokenized_input = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt").to(model.device)

# ancestral sampling で 5 回サンプリング
num_samples = 5
samples = []

with torch.no_grad():
    for _ in range(num_samples):
        output = model.generate(
            tokenized_input,
            max_new_tokens=5, # 生成するトークン数
            temperature=2.0, # 温度パラメータ
            do_sample=True,
        )
        samples.append(output)

for i, sample in enumerate(samples):
    print(f"Sample {i+1}: {tokenizer.decode(sample[0])}")

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:7 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to 

Sample 1: 明日の記憶をDVD(
Sample 2: 明日はどっちだ<EOD|LLM-jp>
Sample 3: 明日が見たいから【
Sample 4: 明日は第4回京都
Sample 5: 明日は日曜日朝まで仕事
